## Lab 6: Classification fine-tuning
**Group:** Phillip Graf, Konstantin Schmidt, Fabian Holländer

In this lab, we import our classification dataset, perform necessary preprocessing steps and the train the OpenAI Gpt2 model, such that it can label reciepes into 9 different categories. The cool thing is that multilabeling is possible. That means if a reciepe is a bakery and vegetable the model classify it as it is.

The 9 different categories are:
.  
.  
TODO Beschreibung vollenden...  
.  
.  



For learning and testing purposes we import a reduced classification dataset of 100.000 annotated reciepes.
https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv

*Optional* the full dataset containing 2 million annotated reciepes and can be used as input for larger-scale training.
https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv

## Importieren notwendiger Python Bibliotheken

In [30]:
import pandas as pd
import os
import re
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version
from utils.gpt_download import download_and_load_gpt2
from utils.gpt import GPTModel, load_weights_into_gpt
from utils.gpt import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)
torch.manual_seed(123)

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

tokenizer = tiktoken.get_encoding("gpt2")

matplotlib version: 3.10.8
numpy version: 2.0.2
tiktoken version: 0.9.0
torch version: 2.10.0
tensorflow version: 2.21.0
pandas version: 2.3.3


## Laden und vorbereiten des Datensatzes

Der Datensatz hat eigentlich 5 Spalten... wir bereiten diese so auf, dass wir nur noch ein Text haben und 2 Classifzierungsspalten. Da im späteren Verlauf ein LLM auch das Gesamte Rezept als Input nimmt und nicht aufgeteilt in verschiedene Spalten.

Wir lassen uns ausgeben welche Label wir haben und wieviele Zeilen pro Label und bringen den Datensatz danach ins Gleichgewicht, damit ein label, dass Training nicht verzerrt oder ein Label nicht untereprsäentiert ist. Deshalb undersampeln wir hier.

Inklusive auf splitten in Traingings und Validierungs und test datensatz

### Design-Entscheidung: Textstrukturierung & Tokenisierung

**Überlegung:**
Ursprünglich wurde in Erwägung gezogen, die Abschnitte der Rezepte (Titel, Zutaten, Anleitung) über explizite Steuerungstokens (*Special Control Tokens* wie `<|startofingredients|>`) zu trennen. Dies ist architektonisch sauber, da das Modell so die semantischen Grenzen im Text exakt erlernen kann.

**Gewählte Lösung & Begründung:**
Wir haben uns letztlich gegen eigene Special Tokens und **für die Strukturierung mittels klassischer Zeilenumbrüche (`\n`)** entschieden. 

* **Problem bei Custom Tokens:** Da wir ein **vortrainiertes GPT-2-Modell** nutzen, würde das Hinzufügen neuer Tokens das originale Vokabular verändern. Die Gewichte des Embedding-Layers würden nicht mehr passen (`Shape Mismatch`), und das Modell müsste für die neuen Tokens komplett neue Vektoren von null auf lernen.
* **Vorteil von `\n`:** GPT-2 wurde auf riesigen Textmengen trainiert und hat bereits verinnerlicht, dass Zeilenumbrüche (`\n`) gefolgt von Schlüsselwörtern wie `Ingredients:` oder `Directions:` Abschnitte strukturieren. Wir nutzen dieses vortrainierte Wissen (*Prior Knowledge*) aus, um das Fine-Tuning stabiler und effizienter zu gestalten.

In [26]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv"
# Uncomment the following line to use the full annotated dataset
# cloud_url = "https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv"

try:
    print("Datensatz aus der Cloud laden...")
    df = pd.read_csv(cloud_url)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    print("Datensatz erfolgreich geladen!\n")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")

def combine_recipe_features(row):
    title = str(row['title']).strip()
    ingredients = str(row['NER']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    return f"Recipe: {title}\nIngredients: {ingredients}\nDirections: {directions}"

print("Bereite Datensatz für das Classification-Fine-Tuning vor...")

df['full_recipe'] = df.apply(combine_recipe_features, axis=1)
df = df[['full_recipe', 'genre', 'label']]

print("Aufbereitung erfolgreich abgeschlossen!\n")
df

Datensatz aus der Cloud laden...
Datensatz erfolgreich geladen!

Bereite Datensatz für das Classification-Fine-Tuning vor...
Aufbereitung erfolgreich abgeschlossen!



,full_recipe,genre,label
0,Recipe: Reeses Cups(Candy)\nIngredients: peanu...,drinks,2
1,Recipe: Rhubarb Coffee Cake\nIngredients: suga...,drinks,2
2,Recipe: Quick Barbecue Wings\nIngredients: chi...,nonveg,3
3,Recipe: Chocolate Frango Mints\nIngredients: c...,drinks,2
4,Recipe: Corral Barbecued Beef Steak Strips\nIn...,drinks,2
...,...,...,...
99995,Recipe: Basic Marinated And Baked Tofu\nIngred...,nonveg,3
99996,Recipe: Bouranee Baunjan - Afghan Eggplant Wit...,sides,8
99997,Recipe: Ginger-Soy-Lime Marinated Shrimp\nIngr...,drinks,2
99998,Recipe: Coffee Can Pumpkin Bread\nIngredients:...,cereal,6


In [ ]:
# Anzeige der Anzahl der Rezepte pro Genre und Label
summary_df = df.groupby(['label', 'genre']).size().reset_index(name='count').sort_values('label')
print(summary_df.to_string(index=False))

 label      genre  count
     1     bakery   4112
     2     drinks  21293
     3     nonveg  14714
     4 vegetables  20200
     5   fastfood   6124
     6     cereal  16803
     7       meal   1687
     8      sides  11832
     9     fusion   3235


In [ ]:
# Undersampling um ein gleiches Verhältnis der Klassen zu erreichen
def create_balanced_dataset(df):
    min_size = df['label'].value_counts().min()
    
    subsets = []
    for label in df['label'].unique():
        label_subset = df[df['label'] == label].sample(min_size, random_state=123)
        subsets.append(label_subset)
        
    balanced_df = pd.concat(subsets).reset_index(drop=True)
    return balanced_df

balanced_df = create_balanced_dataset(filtered_df)
print(balanced_df.groupby(['label', 'genre']).size().reset_index(name='count').to_string(index=False))

 label      genre  count
     1     bakery   1685
     2     drinks   1685
     3     nonveg   1685
     4 vegetables   1685
     5   fastfood   1685
     6     cereal   1685
     7       meal   1685
     8      sides   1685
     9     fusion   1685


In [ ]:
# Aufteilen in Trainings-, Validierungs- und Testset
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

os.makedirs("classification", exist_ok=True)

train_df.to_csv("classification/train.csv", index=None)
validation_df.to_csv("classification/validation.csv", index=None)
test_df.to_csv("classification/test.csv", index=None)

## Erstellen eines spezifischen Dataloader

Da die Rezepte unterschiedlich lang sind, müssen wir sie für das Batch-Training vereinheitlichen. Dafür füllen wie allte Texte auf die länge des längsten Textes auf (Padding). Als Padding-Token nutzen wir das <|endoftext|> Token.

Training, Validierung und Testdatensatz padden.
- Note that validation and test set samples that are longer than the longest training example are being truncated via `encoded_text[:self.max_length]` in the `SpamDataset` code
- This behavior is entirely optional, and it would also work well if we set `max_length=None` in both the validation and test set cases

In [40]:
class RecipeDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["full_recipe"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(shifted_label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

    def _shortest_encoded_length(self):
        if not self.encoded_texts:
            return 0
        min_length = float('inf')
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length < min_length:
                min_length = encoded_length
        return min_length

In [34]:
tokenizer = tiktoken.get_encoding("gpt2")

In [57]:
files = {
    "Train": "classification/train.csv",
    "Validation": "classification/validation.csv",
    "Test": "classification/test.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)
    lengths = [len(tokenizer.encode(text)) for text in df["full_recipe"]]
    
    # Text vorab zusammenbauen
    min_str = f"{min(lengths)} Token"
    max_str = f"{max(lengths)} Token"
    
    # Jetzt den kompletten Text blockweise ausrichten
    print(f"{name:<10} | Kürzestes Rezept: {min_str:<4} | Längstes Rezept: {max_str}")

Train      | Kürzestes Rezept: 22 Token | Längstes Rezept: 982 Token
Validation | Kürzestes Rezept: 28 Token | Längstes Rezept: 833 Token
Test       | Kürzestes Rezept: 30 Token | Längstes Rezept: 879 Token


In [59]:
train_dataset = RecipeDataset(
    csv_file="classification/train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

# Schneide die Texte im Validierungs- und Testset auf die maximale Länge des Trainingssets zu
val_dataset = RecipeDataset(
    csv_file="classification/validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(val_dataset.max_length)

test_dataset = RecipeDataset(
    csv_file="classification/test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(test_dataset.max_length)

982
982
982
